# Prioritized Experience Replay — Schaul et al. (2015)

_Papers · Reinforcement Learning_

**Replay the transitions you learn most from, not just the ones you happened to see.**

---

This notebook is a practice scaffold for the **Prioritized Experience Replay — Schaul et al. (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Recompute the paper's priority → probability → IS-weight pipeline

Before any training, we replay the lesson's worked example by hand so the three core equations are concrete. Each transition gets a priority `p_i = |TD-error| + eps`; Eqn. 1 turns priorities into sampling probabilities `P(i) = p_i^alpha / sum_k p_k^alpha`; and the importance-sampling weight `w_i = (1/(N·P(i)))^beta` corrects the bias that non-uniform sampling introduces. We normalize the weights by their max so they only ever scale updates *down*.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import gym   # CartPole; preinstalled in Colab.

torch.manual_seed(0)
np.random.seed(0)

# Hyperparameters for the worked example.
eps = 0.01
alpha = 0.6
beta = 0.4
N = 3

# |TD-errors| for three transitions.
d = np.array([2.0, 1.0, 0.5])

# Priority of each transition: p_i = |delta_i| + eps.
p = np.abs(d) + eps

# Eqn. 1 — turn priorities into sampling probabilities.
p_pow = p ** alpha
P = p_pow / p_pow.sum()

# Raw importance-sampling weight: w_i = (1 / (N P(i)))^beta.
w = (1.0 / (N * P)) ** beta

# Normalize by the max weight so weights only ever scale updates DOWN.
w_norm = w / w.max()

print("P     =", np.round(P, 4).tolist())        # [0.476, 0.315, 0.209]
print("w/max =", np.round(w_norm, 4).tolist())   # [0.7195, 0.8487, 1.0]

# Sanity: with alpha=0 sampling is uniform (P=1/N) and every weight is exactly 1.
uniform_weight = (1.0 / (N * (1.0 / N))) ** beta
print("uniform w =", float(uniform_weight))      # 1.0

### Step 2 — Build the prioritized replay buffer

The buffer stores transitions in a ring and tracks a priority array alongside them. A *new* transition is inserted at the current max priority so it is guaranteed to be replayed at least once. `sample` implements Eqn. 1 to draw a batch, computes the matching IS-weights, and `update` writes the fresh `|delta| + eps` priorities back after learning. Setting `alpha=0` collapses the whole scheme to uniform sampling — that is the ablation we compare against later.

In [ ]:
class PrioritizedReplay:
    def __init__(self, cap, alpha=0.6, eps=1e-2):
        self.cap = cap
        self.alpha = alpha
        self.eps = eps
        self.data = []
        self.prio = np.zeros(cap, dtype=np.float32)
        self.pos = 0

    def add(self, *transition):
        # A new transition enters at the current MAX priority so it is replayed at least once.
        max_prio = self.prio.max() if self.data else 1.0
        if len(self.data) < self.cap:
            self.data.append(transition)
        else:
            self.data[self.pos] = transition
        self.prio[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.cap

    def sample(self, batch, beta):
        n = len(self.data)
        prio_pow = self.prio[:n] ** self.alpha     # p_i^alpha  (alpha=0 -> all ones -> uniform)
        probs = prio_pow / prio_pow.sum()          # Eqn. 1
        idx = np.random.choice(n, batch, p=probs)
        weights = (n * probs[idx]) ** (-beta)      # w_i = (N P(i))^(-beta)
        weights = weights / weights.max()          # normalize -> downscale only
        sampled = [self.data[i] for i in idx]
        return idx, sampled, torch.tensor(weights, dtype=torch.float32)

    def update(self, idx, td):
        # Write |delta| + eps back as the new priorities.
        self.prio[idx] = np.abs(td) + self.eps


def qnet(obs, act):
    return nn.Sequential(nn.Linear(obs, 128), nn.ReLU(), nn.Linear(128, act))

### Step 3 — The DQN training loop with prioritized sampling

This is a standard DQN over CartPole, with one twist: each gradient step samples a batch *by priority*, weights the TD-error squared by the IS-weights, and feeds the new TD-errors back as priorities. `beta` is annealed from 0.4 toward 1.0 over training so the bias correction strengthens as the policy stabilizes. The same function runs the uniform ablation by passing `alpha=0`.

In [ ]:
def train(prioritized, episodes=200, gamma=0.99):
    torch.manual_seed(0)
    np.random.seed(0)
    env = gym.make("CartPole-v1")
    o_dim = env.observation_space.shape[0]
    a_dim = env.action_space.n

    # Online network q and target network qt (kept in sync periodically).
    q = qnet(o_dim, a_dim)
    qt = qnet(o_dim, a_dim)
    qt.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=1e-3)

    # alpha=0 turns the buffer into a uniform-sampling ablation.
    buf = PrioritizedReplay(10000, alpha=(0.6 if prioritized else 0.0))
    eps_greedy = 1.0
    returns = []

    for ep in range(episodes):
        s, _ = env.reset(seed=ep)
        done = False
        R = 0.0
        beta = 0.4 + 0.6 * ep / episodes          # anneal beta 0.4 -> 1.0

        while not done:
            # Epsilon-greedy action selection.
            if np.random.rand() < eps_greedy:
                a = env.action_space.sample()
            else:
                a = int(q(torch.tensor(s, dtype=torch.float32)).argmax())

            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            buf.add(s, a, r, s2, done)
            s = s2
            R += r

            if len(buf.data) >= 64:
                idx, batch, weights = buf.sample(64, beta)

                # Unpack the sampled batch into tensors.
                S = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
                A = torch.tensor([b[1] for b in batch])
                Rw = torch.tensor([b[2] for b in batch], dtype=torch.float32)
                S2 = torch.tensor(np.array([b[3] for b in batch]), dtype=torch.float32)
                D = torch.tensor([b[4] for b in batch], dtype=torch.float32)

                # Q(s,a) for the taken actions.
                qsa = q(S).gather(1, A[:, None]).squeeze(1)

                # One-step TD target from the frozen target network.
                with torch.no_grad():
                    tgt = Rw + gamma * qt(S2).max(1).values * (1 - D)

                td = tgt - qsa
                loss = (weights * td.pow(2)).mean()   # IS-weighted loss

                opt.zero_grad()
                loss.backward()
                opt.step()

                # Priorities <- |delta| + eps for the sampled transitions.
                buf.update(idx, td.detach().numpy())

        eps_greedy = max(0.05, eps_greedy * 0.97)
        if ep % 5 == 0:
            qt.load_state_dict(q.state_dict())
        returns.append(R)

    env.close()
    return returns


print("\nPRIORITIZED replay:")
pr = train(prioritized=True)
print("UNIFORM replay (ablation, alpha=0):")
un = train(prioritized=False)
print("\nprioritized mean return (last 50 eps):", round(np.mean(pr[-50:]), 1))
print("uniform     mean return (last 50 eps):", round(np.mean(un[-50:]), 1))
# Prioritized typically reaches high return in fewer episodes than uniform.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does prioritizing high-TD-error transitions learn with fewer updates than uniform replay -- and does the gap grow with task length?_

### Step 1 — Build the Blind Cliffwalk environment

The paper's Blind Cliffwalk (Sec 2 / Fig 1) is a chain of `n` states. In each state the `right` action advances one step and the `wrong` action terminates with zero reward; only reaching the final state pays out reward 1. With `gamma = 1` the lone reward must propagate all the way back through the chain, which makes the order in which transitions are replayed matter enormously. We collect every transition once into a fixed buffer.

In [ ]:
import numpy as np

def build(n):
    # Chain of n states; only the last 'right' transition is rewarding.
    t = []
    for s in range(n):
        # 'wrong' action (a=0): terminate immediately with reward 0.
        t.append((s, 0, 0.0, None, True))
        # 'right' action (a=1): advance; the last state pays reward 1.
        if s == n - 1:
            t.append((s, 1, 1.0, None, True))
        else:
            t.append((s, 1, 0.0, s + 1, False))
    return t

### Step 2 — Run tabular Q-learning under uniform vs. priority sampling

We do exact tabular updates (learning rate 1) and count how many updates it takes to reach the optimal Q-values `Q*(s, right) = 1`. `uniform` draws a transition blindly; `priority` uses an oracle that always replays the transition with the current largest `|TD-error|`. Because only one transition carries signal at a time, the priority oracle chains the reward backward optimally.

In [ ]:
def run(n, mode, seed=0, gamma=1.0, max_updates=10_000_000):
    rng = np.random.default_rng(seed)
    trans = build(n)
    Q = np.zeros((n, 2))

    def td(t):
        s, a, r, sn, done = t
        target = r if done else r + gamma * Q[sn].max()
        return target - Q[s, a]

    # Optimal table: Q*(s, right) = 1, Q*(s, wrong) = 0.
    Qstar = np.zeros((n, 2))
    Qstar[:, 1] = 1.0

    for u in range(1, max_updates + 1):
        if mode == "uniform":
            i = rng.integers(len(trans))                 # sample blindly
        else:
            errs = np.abs([td(t) for t in trans])        # priority oracle: max |TD-error|
            i = int(rng.choice(np.flatnonzero(errs >= errs.max() - 1e-12)))

        t = trans[i]
        Q[t[0], t[1]] += td(t)                           # exact tabular update (lr=1)

        if np.max(np.abs(Q - Qstar)) < 1e-9:
            return u

### Step 3 — Compare updates-to-converge as the chain grows

We sweep the chain length `n` and take the median updates-to-converge over many seeds for each sampling mode. Uniform sampling needs a number of updates that grows steeply with `n`, while the priority oracle needs exactly `n` updates — one per link in the chain. The speedup ratio grows with `n`, reproducing the paper's exponential Blind-Cliffwalk effect.

In [ ]:
ns = [4, 6, 8, 10, 12, 14, 16]
unif = [int(np.median([run(n, "uniform",  s) for s in range(15)])) for n in ns]
prio = [int(np.median([run(n, "priority", s) for s in range(15)])) for n in ns]

print("uniform     :", [[n, u] for n, u in zip(ns, unif)])
print("prioritized :", [[n, p] for n, p in zip(ns, prio)])
print("speedup     :", [round(u / p, 1) for u, p in zip(unif, prio)])
# uniform     -> 31, 64, 105, 156, 274, 384, 471   (grows fast)
# prioritized -> n exactly:  4, 6, 8, 10, 12, 14, 16
# speedup grows with n (~7x -> ~28x): the paper's exponential Blind-Cliffwalk effect.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working prioritized DQN on a sparse-reward task (only the final
            transition gives reward). You flip alpha from $0.6$ to $0$ and retrain with everything
            else identical. What happens to learning speed, and what does that isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set alpha = 0. Now $p_i^0 = 1$ for every transition, so $P(i) = 1/N$ &mdash; the buffer samples uniformly. — _$\alpha$ is the single knob between prioritized and uniform; turning it to $0$ recovers the original DQN replay exactly._
- Retrain and watch updates-to-converge: uniform needs many more, because the one informative (reward-reaching) transition is now sampled as rarely as every boring one. — _Uniform spends most updates on already-learned transitions; the rare high-TD-error transition that carries the signal is diluted._
- Note the gap grows with task length (more states to chain the reward back through). — _Each extra state multiplies the number of uniform draws needed to hit the right transition in the right order &mdash; the paper's exponential Blind-Cliffwalk effect._

**Answer:** With $\alpha = 0$ the buffer is uniform and learning is much slower &mdash; and the slowdown
                 worsens as the task lengthens. Since only $\alpha$ changed, this isolates prioritized
                 sampling (not network size, optimizer, or data) as the cause of the speed-up. The CODEVIZ
                 panel reproduces exactly this contrast on the Blind Cliffwalk.

</details>

**Problem 2.** Recompute the pipeline for a fresh batch. Buffer of $N = 2$ transitions with $|\delta| = [3.0,\;
            1.0]$, $\epsilon = 0$, $\alpha = 0.5$, $\beta = 1$. Find $P(i)$ and the normalized IS-weights.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Priorities (no floor): $p = [3.0,\; 1.0]$. Raise to $\alpha = 0.5$ (square root): $p^{0.5} = [\sqrt{3},\; \sqrt{1}] = [1.732,\; 1.0]$, sum $= 2.732$. — _$P(i) \propto p_i^{\alpha}$ needs the powered priorities and their sum (Eqn. 1)._
- Probabilities: $P = [1.732/2.732,\; 1.0/2.732] = [0.634,\; 0.366]$. — _Normalize so they sum to $1$._
- Raw weights $w_i = (1/(N\,P(i)))^{\beta} = (1/(2\,P(i)))^{1}$: $w = [1/(2\cdot0.634),\; 1/(2\cdot0.366)] = [0.789,\; 1.366]$. — _$\beta = 1$ is full correction, so $w_i = 1/(N P(i))$ directly._
- Normalize by $\max w = 1.366$: $w/\max = [0.577,\; 1.0]$. — _Largest weight becomes $1$; updates only scale down._

**Answer:** $P = [0.634,\; 0.366]$; normalized IS-weights $= [0.577,\; 1.0]$. The high-error transition
                 is sampled more (P = 0.634) but its update is scaled down most (weight 0.577) &mdash;
                 prioritize the sampling, then undo the resulting over-count in the gradient.

</details>

**Problem 3.** A teammate removes the $\epsilon$ floor "to simplify," using $p_i = |\delta_i|$. After a while
            some transitions stop being replayed entirely, even ones that later turn out to be mispredicted.
            Why, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find a transition the network currently predicts perfectly: $\delta = 0 \Rightarrow p_i = |0| = 0$. — _Priority is the absolute TD-error, which can hit exactly $0$._
- Its sampling probability $P(i) = p_i^{\alpha} / \sum_k p_k^{\alpha} = 0$, so it is never drawn again. — _A zero priority gives zero probability under Eqn. 1 &mdash; permanent exclusion._
- If the network later drifts and that transition becomes wrong, its priority is still recorded as $0$, so it stays frozen out and the error is never corrected. — _Priorities only update when a transition is sampled; a never-sampled transition can never refresh its stale $0$._
- Fix: restore the floor, $p_i = |\delta_i| + \epsilon$ with small $\epsilon$. — _A nonzero floor guarantees every transition keeps a small but positive chance of being revisited._

**Answer:** Without $\epsilon$, any transition reaching $\delta = 0$ gets priority $0$ and probability
                 $0$, so it is frozen out permanently &mdash; and can never refresh its priority even if it
                 later becomes mispredicted. The $\epsilon$ floor ($p_i = |\delta_i| + \epsilon$) keeps every
                 transition reachable.

</details>